# 02 — Advanced RAG Techniques

**Goal**: Implement HyDE, Multi-Query, and Reranking to improve RAG quality.

In [ ]:
import chromadb
import requests
import numpy as np

client = chromadb.PersistentClient(path="./chroma_rag")

def get_embedding(text):
    resp = requests.post("http://localhost:11434/api/embeddings", json={
        "model": "nomic-embed-text",
        "prompt": text
    })
    return np.array(resp.json()["embedding"])

def ollama_generate(prompt):
    resp = requests.post("http://localhost:11434/api/generate", json={
        "model": "llama3.2",
        "prompt": prompt,
        "stream": False,
        "temperature": 0.1
    })
    return resp.json()["response"]

# Reuse knowledge base from previous notebook
collection = client.get_or_create_collection(name="knowledge_base")
print(f"Docs in collection: {collection.count()}")

## 1. Technique 1: HyDE (Hypothetical Document Embeddings)

Problem: User query "How do I fine-tune?" doesn't look like a document.

Solution: Generate a hypothetical answer first, use THAT for retrieval.

In [ ]:
def hyde_search(query, top_k=2):
    # Step 1: Generate a hypothetical document
    hypo_prompt = f"Write a paragraph that would answer the question: {query}"
    hypothetical_doc = ollama_generate(hypo_prompt)
    
    # Step 2: Use the hypothetical doc for retrieval (instead of the query)
    hypo_emb = get_embedding(hypothetical_doc)
    results = collection.query(
        query_embeddings=[hypo_emb],
        n_results=top_k
    )
    
    # Step 3: Generate final answer using real retrieved docs
    context = "\n\n".join(results['documents'][0])
    final_prompt = f"""Answer the question using the context.

Context: {context}

Question: {query}
Answer:"""
    answer = ollama_generate(final_prompt)
    
    return answer, results['documents'][0], hypothetical_doc[:100]

answer, sources, hypo = hyde_search("How do I fine-tune a model efficiently?")
print(f"Hypothetical doc: {hypo}...")
print(f"\nRetrieved: {sources}")
print(f"\nAnswer: {answer}")

## 2. Technique 2: Multi-Query Retrieval

Generate multiple variations of the query and search with each.

In [ ]:
def multi_query_search(original_query, n_queries=3, top_k_per_query=2):
    # Step 1: Generate query variations
    prompt = f"""Generate {n_queries} different versions of this question to help find relevant documents.
Original: {original_query}
Return as a list, one per line, no numbering."""
    
    variations_text = ollama_generate(prompt)
    variations = [v.strip() for v in variations_text.split("\n") if v.strip()]
    variations = variations[:n_queries]
    
    print(f"Query variations:")
    for v in variations:
        print(f"  - {v}")
    
    # Step 2: Search with each variation
    all_results = []
    for v in [original_query] + variations:
        query_emb = get_embedding(v)
        results = collection.query(
            query_embeddings=[query_emb],
            n_results=top_k_per_query
        )
        all_results.extend(results['documents'][0])
    
    # Step 3: Deduplicate and take top results
    seen = set()
    unique_results = []
    for doc in all_results:
        if doc not in seen:
            seen.add(doc)
            unique_results.append(doc)
    
    context = "\n\n".join(unique_results[:3])
    final_prompt = f"""Answer using the context.

Context: {context}

Question: {original_query}
Answer:"""
    answer = ollama_generate(final_prompt)
    
    return answer, unique_results

answer, sources = multi_query_search("Tell me about quantization methods")
print(f"\nRetrieved: {sources}")
print(f"\nAnswer: {answer}")

## 3. Technique 3: Reranking

Retrieve more docs than needed, then re-rank by relevance using a cross-encoder.

In [ ]:
# !pip install sentence-transformers
from sentence_transformers import CrossEncoder

# Cross-encoder is more accurate than cosine similarity for relevance scoring
# but slower — we use it to re-rank the top-N results
reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

def rerank_search(query, top_k_retrieve=10, top_k_final=3):
    # Step 1: Retrieve more docs
    query_emb = get_embedding(query)
    results = collection.query(
        query_embeddings=[query_emb],
        n_results=top_k_retrieve
    )
    
    # Step 2: Rerank with cross-encoder
    pairs = [[query, doc] for doc in results['documents'][0]]
    scores = reranker.predict(pairs)
    
    # Step 3: Sort by new scores
    ranked = sorted(zip(results['documents'][0], scores), key=lambda x: x[1], reverse=True)
    
    print(f"Reranked results:")
    for doc, score in ranked[:top_k_final]:
        print(f"  [{score:.3f}] {doc}")
    
    context = "\n\n".join([doc for doc, _ in ranked[:top_k_final]])
    final_prompt = f"""Answer using the context.

Context: {context}

Question: {query}
Answer:"""
    answer = ollama_generate(final_prompt)
    return answer

answer = rerank_search("How does RAG work with vector databases?")
print(f"\nAnswer: {answer}")

## 4. Putting It All Together: Advanced RAG Pipeline

The production RAG pipeline:

In [ ]:
def advanced_rag(query):
    # 1. HyDE: Generate hypothetical doc
    hypo = ollama_generate(f"Write a paragraph answering: {query}")
    
    # 2. Multi-query expansion
    variations = ollama_generate(f"Rephrase this question 3 ways: {query}")
    queries = [query] + [v.strip() for v in variations.split("\n") if v.strip()][:2]
    
    # 3. Retrieve with combined embeddings
    all_docs = []
    for q in queries + [hypo]:
        emb = get_embedding(q)
        results = collection.query(query_embeddings=[emb], n_results=5)
        all_docs.extend(results['documents'][0])
    
    # 4. Dedup
    unique = list(dict.fromkeys(all_docs))
    
    # 5. Rerank (if cross-encoder available)
    try:
        pairs = [[query, d] for d in unique[:20]]
        scores = reranker.predict(pairs)
        ranked = sorted(zip(unique[:20], scores), key=lambda x: x[1], reverse=True)
        final_docs = [d for d, _ in ranked[:3]]
    except:
        final_docs = unique[:3]
    
    # 6. Generate
    context = "\n\n".join(final_docs)
    return ollama_generate(f"Context: {context}\n\nQuestion: {query}\nAnswer:"), final_docs

answer, sources = advanced_rag("Compare LoRA and QLoRA")
print(f"Answer: {answer}")

## Key Takeaways

1. **HyDE**: Generate hypothetical answer → use as search query
2. **Multi-Query**: Search with query variations → merge results
3. **Reranking**: Cross-encoder scores relevance better than embedding similarity
4. **Combine techniques** for production-quality RAG

**Next**: Evaluate RAG quality →